# Entity Resolution & Hyperparameter Optimization

This notebook outlines a systematic process for optimizing our **Entity Disambiguation** pipeline. The goal is to identify the most accurate configuration for grouping person mentions by comparing localized model outputs against their corresponding ground truth data.

## Methodology

### 1. Data Correspondence & Ground Truth
To ensure data integrity, we have moved away from a single global file approach. The pipeline now maintains a strict 1:1 correspondence between files to prevent entity overlaps across different documents. The evaluation process compares:

* **`gold_json`**: The benchmark "Gold Standard" for a specific document.
* **`input_ner_preds_json`**: The raw NER model predictions for that specific input.
* **`cluster_json`**: The output generated by our clustering logic for those specific predictions.

### 2. NER Extraction & Processing
We deploy our **Named Entity Recognition (NER)** model across the source documents. Due to performance constraints on macOS when processing large batches in a single loop, we process documents individually. 

This sequential processing allows us to:
1. Effectively utilize cache memory.
2. Identify and fix incomplete entities (e.g., missing `entity_id`) before the evaluation phase.
3. Ensure each `input_ner_preds_json` is correctly mapped to its `gold_json`.

### 3. Clustering via `cluster_with_cdist`
The core of our pipeline is the `cluster_with_cdist` function, which transforms raw NER predictions into grouped clusters:

* **Similarity Matrix**: Uses `process.cdist` from `rapidfuzz` to calculate an efficient similarity matrix based on a specific `scorer`, `score_cutoff threshold` and `processor`.
* **Union-Find Algorithm**: Implements **Disjoint Set Union (DSU)** logic to link entities that exceed the similarity threshold.
* **Canonical Selection**: Through `pick_canonical`, the system selects the most representative string for each group, prioritizing the longest original string.

### 4. Systematic Grid Search & Evaluation
We execute a **Grid Search** by iterating through clustering hyperparameters (`Scorers`, `Thresholds`, and `Processors`). For each iteration, the resulting `cluster_json` is validated against its correspondent `gold_json`.

#### Evaluation Metric: Weighted Disambiguation Score
The performance of each hyperparameter combination is measured by a composite score that balances four key dimensions:

* **Entity-level F1**: Measures the overall quality of entity discovery.
* **Alias Macro F1**: Evaluates how well different name variants (aliases) are grouped.
* **Label Accuracy**: Ensures the primary name assigned to the cluster is correct.
* **Role Accuracy**: Validates the secondary attributes (roles) associated with the entity.

#### Winner Selection
To ensure the robustness of our pipeline, the **winning combination** is determined by calculating the **best average score across all documents**. This cross-document averaging prevents overfitting to a single file and ensures that the selected `scorer`, `threshold` and `processor` perform consistently across the entire corpus.

---

> **Implementation Note:** By analyzing the weighted score across discrete file correspondences, we can determine the optimal configuration while avoiding the data pollution issues found in previous single-file versions.

In [ ]:
%load_ext rich

%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

import json
import mimetypes
import os
import re
import time
import unicodedata
from collections import Counter
from operator import itemgetter
from pathlib import Path
from typing import Iterable, Tuple

import requests
from tqdm import tqdm

import numpy as np
import pandas as pd
import requests
from more_itertools import flatten, unique_everseen

from aymurai.meta.entities import CanonicalEntities, CanonicalEntity
from aymurai.utils.json_data import get_pretty, save_json, load_json
import aymurai.evaluation.metrics as aymurai_metrics

# #**1** First Step: Prepare the ***gold_json*** and ***ner_preds_json***

## /document-extract endpoint output

In [ ]:
API_URL = os.getenv("DOCUMENT_API_BASE_URL", "http://127.0.0.1:8999")
ENDPOINT = f"{API_URL}/misc/document-extract"
DATA_ROOT = Path(
    os.getenv(
        "DOCUMENT_DATA_ROOT", "../../../resources/data/restricted/disambiguation-eval/files"
    )
)
GOLD_JSON_ROOT = Path(
    os.getenv(
        "GOLD_JSON_ROOT",
        "../../../resources/data/restricted/disambiguation-eval/canonical-entities/manual-predicted",
    )
)
DOC_EXTENSIONS = {".pdf", ".docx"}
JSON_EXTENSION = {".json"}
REQUEST_TIMEOUT_S = float(os.getenv("DOCUMENT_REQUEST_TIMEOUT", "30"))

print(f"Target endpoint: {ENDPOINT}")
print(f"Data root: {DATA_ROOT.resolve()}")

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
print(f"Discovered {len(documents)} documents.")

gold_jsons = discover_documents(GOLD_JSON_ROOT, JSON_EXTENSION)
print(f"Discovered {len(gold_jsons)} gold JSON files.")

In [ ]:
def call_extraction_api(
    session: requests.Session, file_path: Path
) -> dict[str, object]:
    payload: dict[str, object] = {
        "path": str(file_path),
        "status": "failure",
        "status_code": None,
        "elapsed_s": None,
        "detail": None,
    }

    if not file_path.exists():
        payload["detail"] = "File does not exist"
        return payload

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    files = {
        "file": (file_path.name, file_path.open("rb"), mime_type),
    }

    try:
        start = time.perf_counter()
        response = session.post(
            ENDPOINT,
            files=files,
            timeout=REQUEST_TIMEOUT_S,
        )
        elapsed = time.perf_counter() - start
    except requests.RequestException as exc:
        payload["detail"] = f"Request failed: {exc}"
        return payload
    finally:
        files["file"][1].close()

    payload["status_code"] = response.status_code
    payload["elapsed_s"] = elapsed

    try:
        response_body = response.json()
    except ValueError:
        response_body = {"raw": response.text[:500]}

    if response.ok:
        payload["status"] = "success"
        payload["detail"] = {
            "document_id": response_body.get("document_id"),
            "document": response_body.get("document", []),
        }
    else:
        payload["detail"] = response_body

    return payload

## Inference

In [ ]:
from itertools import chain
from operator import itemgetter
from typing import Any

from more_itertools import unique_everseen


# Function to make inference using the API
def get_predictions(sample: str) -> dict:
    response = requests.post(url=f"{API_URL}/anonymizer/predict", json={"text": sample})
    response.raise_for_status()
    return response.json()


def parse_prediction_labels(predictions: list[dict[str, Any]]) -> list[dict[str, str]]:
    """
    Parse prediction labels to extract unique aymurai_label and aymurai_alt_text pairs.

    Args:
        predictions (list[dict[str, Any]]): A list of prediction dictionaries.

    Returns:
        list[dict[str, str]]: A list of dictionaries containing unique aymurai_label and aymurai_alt_text pairs.
    """
    attrs_stream = (
        label.get("attrs") or {}
        for label in chain.from_iterable(pred.get("labels", ()) for pred in predictions)
    )

    unique_pairs = unique_everseen(
        (
            attrs.get("aymurai_label"),
            attrs.get("aymurai_alt_text"),
        )
        for attrs in attrs_stream
        if attrs.get("aymurai_label") and attrs.get("aymurai_alt_text")
    )

    return sorted(
        ({"aymurai_label": label, "text": text} for label, text in unique_pairs),
        key=itemgetter("aymurai_label", "text"),
    )

## ***gold_json*** and ***ner_preds_json*** for input to clusterization developement

In [ ]:
def extract_and_save_entities(
    document_paths: list[Path], output_path: Path, file_ending: str, target_label: str
) -> None:
    """
    Processes a list of documents to extract specific entities and saves
    each result as an individual JSON file.
    It has been optimized to handle both document files and pre-existing JSON files.
    """

    with requests.Session() as session:
        if ".json" not in document_paths[0].suffix:
            for doc_path in tqdm(
                document_paths, desc=f"Extracting {target_label} entities"
            ):
                doc_path = Path(doc_path)

                # 1. API Extraction
                response = call_extraction_api(session, doc_path)
                document_data = response.get("detail", {}).get("document")

                if not document_data:
                    print(f"Warning: No document content found for {doc_path.name}")
                    continue

                # 2. Processing
                raw_predictions = [
                    get_predictions(paragraph) for paragraph in document_data
                ]
                parsed_labels = parse_prediction_labels(raw_predictions)

                # 3. Filtering by dynamic label
                filtered_entities = [
                    item
                    for item in parsed_labels
                    if item.get("aymurai_label") == target_label
                ]

                # 4. Saving individual file
                clean_base_name = re.sub(
                    r"\s+|_", "-", os.path.splitext(os.path.basename(doc_path))[0]
                )
                clean_base_name = re.sub(r"-{2,}", "-", clean_base_name).strip("-")
                clean_name = re.sub(r"-{2,}", "-", clean_base_name)
                file_name = f"{clean_name}{file_ending}"
                save_path = output_path / file_name

                save_json(file_path=save_path, json_data=filtered_entities)
        else:
            for doc_path in tqdm(
                document_paths, desc=f"Extracting {target_label} entities"
            ):
                doc_path = Path(doc_path)

                # 1. Load existing JSON data
                document_data = load_json(doc_path)

                if not document_data:
                    print(f"Warning: No document content found for {doc_path.name}")
                    continue

                # 2. Filtering by dynamic label
                filtered_entities = [
                    item
                    for item in document_data
                    if item.get("aymurai_label") == target_label
                ]

                # 3. Saving individual file
                clean_base_name = re.sub(
                    r"\s+|_", "-", os.path.splitext(os.path.basename(doc_path))[0]
                )
                clean_base_name = re.sub(r"-{2,}", "-", clean_base_name).strip("-")
                clean_name = re.sub(r"-{2,}", "-", clean_base_name)
                file_name = f"{clean_name}{file_ending}"
                save_path = output_path / file_name

                save_json(file_path=save_path, json_data=filtered_entities)

## ***gold-jsons***

In [ ]:
# We save the gold_json in a directory in /canonical-entities/pre-clusterization

if not (
    DATA_ROOT.parent / "canonical-entities" / "pre-clusterization" / "gold-jsons"
).exists():
    os.makedirs(
        DATA_ROOT.parent / "canonical-entities" / "pre-clusterization" / "gold-jsons"
    )

#### We identified that some .json files were incomplete due to missing entity_id fields. Consequently, we implemented a function to resolve this issue.

In [ ]:
def correct_entity_id(json_path: str) -> None:
    """Correct the entity_id field in the given JSON file to ensure it is a hex string."""

    # Load existing canonical entities from JSON file
    canonical_entities = load_json(json_path)

    # Convert reviewed entities back to CanonicalEntity objects
    canonical_entities = [
        CanonicalEntity.model_validate(entity) for entity in canonical_entities
    ]

    # Update entity_id to be a hex string
    canonical_entities = [
        entity.model_dump() | {"entity_id": entity.entity_id.hex}
        for entity in canonical_entities
    ]

    save_json(file_path=json_path, json_data=canonical_entities)

In [ ]:
data = load_json(gold_jsons[0])
data

In [ ]:
correct_entity_id(gold_jsons[0])

#### Now we are ready to proceed with the ***gold_jsons***

In [ ]:
extract_and_save_entities(
    document_paths=gold_jsons,
    output_path=DATA_ROOT.parent
    / "canonical-entities"
    / "pre-clusterization"
    / "gold-jsons",
    file_ending="-gold.json",
    target_label="PER",
)

------------------------ START OF DISCARD SECTION ------------------------

### This was the old version to make the ***gold_json***, all of the entities in a single file.

In [ ]:
# DISCARD
def gold_json_labels(label: str, json_paths: list[Path]) -> list[dict[str, str]]:
    g_json = [
        item
        for json_path in json_paths
        for item in load_json(json_path)
        if item.get("aymurai_label") == label
    ]

    return g_json

In [ ]:
# DISCARD
gold_json = gold_json_labels(label="PER", json_paths=gold_jsons)

save_json(
    file_path=DATA_ROOT.parent
    / "canonical-entities"
    / "pre-clusterization"
    / "gold.json",
    json_data=gold_json,
)

------------------------ END OF DISCARD SECTION ------------------------

## ***ner-preds-jsons***

#### Due to performance issues on macOS when processing all documents in a single loop, we processed them individually first to ensure they were correctly stored in the cache.

In [ ]:
# Extract document
session = requests.Session()
document = call_extraction_api(session, Path(documents[17]))

In [ ]:
document = document.get("detail", {}).get("document")

if not document:
    raise ValueError("Document text is empty or not found.")

In [ ]:
document

In [ ]:
# We save the gold_json in a directory in /canonical-entities/pre-clusterization

if not (
    DATA_ROOT.parent / "canonical-entities" / "pre-clusterization" / "ner-preds-jsons"
).exists():
    os.makedirs(
        DATA_ROOT.parent
        / "canonical-entities"
        / "pre-clusterization"
        / "ner-preds-jsons"
    )

#### Now we are ready to proceed with the ***ner_preds_json***

In [ ]:
extract_and_save_entities(
    document_paths=documents,
    output_path=DATA_ROOT.parent
    / "canonical-entities"
    / "pre-clusterization"
    / "ner-preds-jsons",
    file_ending="-ner-preds.json",
    target_label="PER",
)

------------------------ START OF DISCARD SECTION ------------------------

#### The following cells belong to a previous version where all NER predictions were stored in a single input file. However, we discovered that certain entities appear in multiple files. Therefore, processing them in a single file would lead to incorrect comparisons against the ***gold_json***.

In [ ]:
# DISCARD
def get_entities_by_label(document_paths: list[Path], target_label: str) -> list[dict]:
    """
    Iterates over a list of documents, extracts entities via API,
    and returns a list of items matching the specified label.
    """
    all_ner_preds = []

    # Using a session to reuse the connection for better performance
    with requests.Session() as session:
        for doc_path in tqdm(document_paths, desc="Processing documents"):
            # API Extraction
            response = call_extraction_api(session, Path(doc_path))
            document_data = response.get("detail", {}).get("document")

            if not document_data:
                print(f"Warning: No document content found for {doc_path}")
                continue

            # Get predictions for each paragraph
            raw_predictions = [
                get_predictions(paragraph) for paragraph in document_data
            ]

            # Parse and flatten labels
            parsed_labels = parse_prediction_labels(raw_predictions)

            # Filter by the selected label and extend the main list
            filtered_entities = [
                item
                for item in parsed_labels
                if item.get("aymurai_label") == target_label
            ]

            all_ner_preds.extend(filtered_entities)

    return all_ner_preds

In [ ]:
# DISCARD
ner_preds_json = get_entities_by_label(document_paths=documents, target_label="PER")

In [ ]:
# DISCARD
save_json(
    file_path=DATA_ROOT.parent
    / "canonical-entities"
    / "pre-clusterization"
    / "ner_preds.json",
    json_data=ner_preds_json,
)

------------------------ END OF DISCARD SECTION ------------------------

## ***Evaluation Pipeline: Ground Truth and NER Prediction Pairing***

We define the following function to ensure a correct implementation of the metrics evaluation for the performance of each combination of hyperparameters. The function returns each pair of .json paths for the clusterization and following evaluation

In [ ]:
def pair_gold_and_preds(
    gold_paths: list[Path], pred_paths: list[Path]
) -> list[Tuple[Path, Path]]:
    """
    Pairs gold standard JSONs with their corresponding NER predictions
    based on the common document prefix.
    """

    # 1. Helper to extract the core ID of the file. It takes everything before "-ner-preds" or "-canonical-entities-gold"
    def get_document_id(path: Path) -> str:
        name = path.stem
        # Remove known suffixes to get the base ID
        name = name.replace("-ner-preds", "")
        name = name.replace("-canonical-entities-gold", "")
        return name

    # 2. Create a mapping of {doc_id: pred_path}
    preds_map = {get_document_id(p): p for p in pred_paths}

    paired_paths = []
    missing_preds = []

    # 3. Iterate through gold files and find their match
    for gold_path in gold_paths:
        doc_id = get_document_id(gold_path)

        if doc_id in preds_map:
            paired_paths.append((gold_path, preds_map[doc_id]))
        else:
            missing_preds.append(gold_path.name)

    # 4. Validation / Logging
    if missing_preds:
        print(f"Warning: No predictions found for {len(missing_preds)} gold files.")
        for missing in missing_preds:
            print(f"   - Missing match for: {missing}")

    print(f"Successfully paired {len(paired_paths)} documents.")

    return paired_paths

In [ ]:
PRE_CLUSTERIZATION_ROOT = GOLD_JSON_ROOT.parent / "pre-clusterization"

gold_docs = discover_documents(
    root=PRE_CLUSTERIZATION_ROOT / "gold-jsons", extensions=JSON_EXTENSION
)
pred_docs = discover_documents(
    root=PRE_CLUSTERIZATION_ROOT / "ner-preds-jsons", extensions=JSON_EXTENSION
)

document_pairs = pair_gold_and_preds(gold_docs, pred_docs)

### **Hyperparameter Grid Search & Evaluation Pipeline**

We define a specialized function to orchestrate a **Grid Search** across multiple hyperparameters. The goal is to systematically evaluate the performance of our clustering model by testing every possible combination of **scorers**, **thresholds**, and **text processors**.

#### **Pipeline Architecture**

1. **Directory-Based Organization**:
For each unique combination of hyperparameters, the function creates a dedicated directory. The folder is named using the parameters (e.g., `scorer-cosine_threshold-0.8_proc-normalized`) to ensure clear traceability.
2. **Pre-Clustering Execution**:
Within each iteration, the pipeline processes the previously paired `gold-json` and `ner-preds-json` files. It generates a new `pre-cluster.json` file, preserving the original document's identity in the filename.
3. **Performance Evaluation**:
Once the cluster is generated, the function evaluates the results against the corresponding **Gold Standard**. This ensures that the metric reflects the actual accuracy of the current hyperparameter configuration.
4. **Iterative Metrics Logging**:
The evaluation results are stored in a centralized JSON file within the specific combination's folder.
* **Keys:** Represent the evaluated document name.
* **Values:** Represent the calculated performance metric.
This file is updated incrementally as the loop progresses through each document pair.


5. **Grid Progression**:
After all document pairs have been processed for a specific set of parameters, the function moves to the next grid cell, repeating the directory creation and evaluation steps until all combinations are exhausted.

In [ ]:
from rapidfuzz import fuzz, process, utils
from rapidfuzz.process import extractOne
from rapidfuzz.fuzz import (
    ratio,
    partial_ratio,
    token_sort_ratio,
    token_set_ratio,
    partial_token_set_ratio,
    partial_token_sort_ratio,
    WRatio,
)

We take the functions defined by Juli in the `04-entity-disambiguation-from-pre-clustered-validations.ipynb` notebook and made one change:
- In `cluster_with_cdist` we added the text processor variable.

In [ ]:
def cluster_with_cdist(
    items: list[dict],
    threshold: int = 90,
    scorer: callable = token_set_ratio,
    processor: callable = None,
):
    """
    Cluster entities and prepare them for CanonicalEntity conversion.
    """
    if not items:
        return []

    # 1. Extract texts and apply normalization
    # We keep track of the original text, the processed text, and the label
    entities = [item.get("text", "") for item in items]
    labels = [item.get("aymurai_label", "UNKNOWN") for item in items]

    if processor:
        normed = [processor(e) for e in entities]
    else:
        normed = [str(e) for e in entities]

    # 2. Similarity Matrix
    sim = process.cdist(normed, normed, scorer=scorer, score_cutoff=threshold)
    sim = np.array(sim)

    # 3. Union-Find Logic
    parent = list(range(len(normed)))

    def find(i):
        if parent[i] == i:
            return i
        parent[i] = find(parent[i])
        return parent[i]

    def union(i, j):
        root_i, root_j = find(i), find(j)
        if root_i != root_j:
            parent[root_j] = root_i

    n = len(normed)
    for i in range(n):
        for j in range(i + 1, n):
            if sim[i, j] >= threshold:
                union(i, j)

    # 4. Group into the format parse_item expects: (orig, norm, label)
    clusters_map = {}
    for idx in range(n):
        root = find(idx)
        if root not in clusters_map:
            clusters_map[root] = []
        # This tuple matches your 'parse_item' len == 3 condition
        clusters_map[root].append((entities[idx], normed[idx], labels[idx]))

    return list(clusters_map.values())


def parse_item(item: tuple[str, ...]) -> tuple[str, str, str]:
    """
    Parse an item into (label, orig, norm).

    Accepts:
      - (orig, norm, label)
      - (labelled_orig, labelled_norm) with prefix 'LABEL:'
    Args:
        item (tuple[str, ...]): input item

    Returns:
        tuple[str, str, str]: (label, orig, norm)
    """
    if len(item) == 3:
        orig, norm, label = item
        return label, orig, norm

    # len == 2: assume "LABEL:text"
    labelled_orig, labelled_norm = item
    label, orig = labelled_orig.split(":", 1)
    _, norm = labelled_norm.split(":", 1)
    return label, orig, norm


def pick_cluster_label(parsed_items: list[tuple[str, str, str]]) -> str:
    """
    Pick the most common label from parsed items.

    Args:
        parsed_items (list[tuple[str, str, str]]): parsed items

    Returns:
        str: chosen label
    """
    labels = [lbl for lbl, _, _ in parsed_items]
    # majority vote; fallback to first
    return Counter(labels).most_common(1)[0][0]


def pick_canonical_text(parsed_items: list[tuple[str, str, str]]) -> str:
    """
    Choose the longest original surface form; tweak as needed.

    Args:
        parsed_items (list[tuple[str, str, str]]): parsed items

    Returns:
        str: chosen canonical text
    """
    return max(parsed_items, key=lambda x: len(x[1]))[1]


def clusters_to_canonical_entities(
    clusters: list[list[tuple[str, str]]],
) -> list[CanonicalEntity]:
    """
    Convert clusters to CanonicalEntity objects.

    Args:
        clusters (list[list[tuple[str, str]]]): clusters of (original, normalized) entity tuples

    Returns:
        list[CanonicalEntity]: list of CanonicalEntity objects
    """
    canonical_entities = []

    for cluster in clusters:
        parsed = [parse_item(item) for item in cluster]  # [(label, orig, norm), ...]
        label = pick_cluster_label(parsed)
        canonical_text = pick_canonical_text(parsed)
        aliases = sorted({orig for _, orig, _ in parsed})
        ce = CanonicalEntity(
            aymurai_label=label,
            canonical_text=canonical_text,
            aliases=aliases,
            attributes={},
            relations=[],
        )
        canonical_entities.append(ce)

    return canonical_entities

We defined 2 new functions:
- `get_name` help us to get the cleaned name of the hyperparameters.
- `run_evaluation_grid_search` to run the grid search of the best hyperparameters combination.


In [ ]:
import shutil

def get_name(obj):
    """Helper to extract a clean string name from a function or object."""
    if obj is None:
        return "None"
    if hasattr(obj, "__name__"):
        return obj.__name__
    # Fallback for complex objects or partials: remove memory addresses
    clean_name = re.sub(r" at 0x[0-9a-fA-F]+", "", str(obj))
    return clean_name.strip("<>").replace("cyfunction ", "").replace("function ", "")


def run_evaluation_grid_search(
    scorers: list,
    thresholds: list,
    processors: list,
    document_pairs: list,
    target_label: str,
    base_output_path: Path,
):
    """
    Runs a grid search over hyperparameters using nested loops.
    """

    # To keep track of all results and find the best
    all_combinations_results = []

    for scorer in scorers:
        for threshold in thresholds:
            for processor in processors:
                # 1. Create a descriptive folder name with the cleaned names
                scorer_name = get_name(scorer)
                processor_name = get_name(processor)
                combo_name = (
                    f"scorer-{scorer_name}-threshold-{threshold}-proc-{processor_name}"
                )

                combo_dir = base_output_path / combo_name
                combo_dir.mkdir(parents=True, exist_ok=True)

                metrics_results = {}
                metrics_file_path = (
                    combo_dir / f"evaluation_metrics_{target_label}.json"
                )

                # List to collect scores for this specific combination
                current_combo_scores = []

                # 2. Iterate over document pairs (Gold vs Prediction)
                for gold_path, pred_path in tqdm(
                    document_pairs, desc=f"Testing {combo_name}"
                ):
                    # Extract entities from the JSON, we filter by target label to ensure consistency if the file has multiple labels
                    entities = load_json(pred_path)
                    filtered_preds = [
                        p for p in entities if p["aymurai_label"] == target_label
                    ]

                    # --- Pre-clustering Logic ---
                    # Run the clustering using your hyperparameters
                    cluster_data = cluster_with_cdist(
                        items=filtered_preds,
                        threshold=threshold,
                        scorer=scorer,
                        processor=processor,
                    )

                    # Convert to Canonical Entities
                    canonical_entities = clusters_to_canonical_entities(cluster_data)

                    canonical_entities = [
                        CanonicalEntity.model_validate(entity)
                        for entity in canonical_entities
                    ]

                    canonical_entities = [
                        entity.model_dump() | {"entity_id": entity.entity_id.hex}
                        for entity in canonical_entities
                    ]

                    # Save the result as the pre-cluster.json
                    cluster_filename = f"{pred_path.stem}-pre-cluster.json"
                    cluster_output_path = combo_dir / cluster_filename

                    save_json(
                        file_path=cluster_output_path, json_data=canonical_entities
                    )

                    # --- Metric Evaluation ---
                    score, metrics = aymurai_metrics.evaluate_disambiguation(
                        gold_json=load_json(gold_path),
                        pred_json=load_json(cluster_output_path),
                        target_label=target_label,
                    )

                    current_combo_scores.append(score)
                    doc_id = pred_path.stem
                    metrics_results[doc_id] = {
                        "label": target_label,
                        "metric_value": score,
                        "detailed_metrics": metrics,
                    }

                # --- Calculate Average for this combination ---
                avg_score = (
                    np.mean(current_combo_scores) if current_combo_scores else 0.0
                )

                # Add average to the results file for this folder
                final_output = {
                    "average_combination_score": avg_score,
                    "document_details": metrics_results,
                }

                with open(metrics_file_path, "w") as f:
                    json.dump(final_output, f, indent=4)

                # Store combination info for final ranking
                all_combinations_results.append(
                    {
                        "name": combo_name,
                        "score": avg_score,
                        "params": {
                            "scorer": scorer_name,
                            "threshold": threshold,
                            "processor": processor_name,
                        },
                    }
                )

    # Save all combinations results in a summary file
    summary_file_path = (
        base_output_path / f"results_summary_{target_label}.json"
    )
    with open(summary_file_path, "w") as f:
        json.dump(all_combinations_results, f, indent=4)

    # --- Find the best combination ---
    best_combo = max(all_combinations_results, key=lambda x: x["score"])

    print(f"\nGrid Search for {target_label} completed.")

    # We now copy the best combination folder to a new location with a standardized name
    src = Path(base_output_path) / best_combo["name"]
    # Combine the parent destination path with the new folder name
    new_name = f"best-pre-clusterization-{target_label.lower()}"
    dest = Path(base_output_path.parent) / new_name
    
    try:
        # copytree creates the destination directory with the 'new_name'
        shutil.copytree(src, dest)
        print(f"Successfully copied '{src.name}' to '{dest}'")
    except FileExistsError:
        print(f"Error: A folder named '{new_name}' already exists in '{base_output_path.parent}'")
    except Exception as e:
        print(f"An error occurred: {e}")    

    return best_combo

In [ ]:
# We have to install jellyfish for the phonetic processor
!pip install jellyfish

In [ ]:
def hard_normalizer(s: str) -> str:
    """
    Normalize string for clustering: strips accents, removes punctuation (.,-),
    lowercases, and collapses spaces.

    Args:
        s (str): input string
    Returns:
        str: normalized string
    """
    if not s:
        return ""

    # Strip accents
    s = "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )

    # Remove commas, periods, and hyphens
    s = re.sub(r"[.,\-]", " ", s)

    # Lowercase and collapse whitespace
    s = " ".join(s.lower().split())

    return s

def light_normalizer(s: str) -> str:
    return s.lower().strip() if s else ""


def legal_text_normalizer(s: str) -> str:
    if not s:
        return ""

    # Standard cleaning (accents/lowercase)
    s = "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )
    s = s.lower()

    # Remove Legal Titles & Prefixes
    legal_titles = r"\b(dr|dra|sr|sra|expte|nro|no|pcia)\b\.?"
    s = re.sub(legal_titles, "", s)

    # Remove common Spanish stopwords
    stopwords = r"\b(de|del|la|las|el|los|y|en)\b"
    s = re.sub(stopwords, "", s)

    # Remove all non-alphanumeric except spaces
    s = re.sub(r"[^\w\s]", "", s)

    return " ".join(s.split())


import jellyfish


def phonetic_normalizer(s: str) -> str:
    if not s:
        return ""
    # Standardize first
    clean = "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    ).lower()

    return jellyfish.nysiis(clean)

In [ ]:
scorers = [
    ratio,
    partial_ratio,
    token_sort_ratio,
    token_set_ratio,
    partial_token_set_ratio,
    partial_token_sort_ratio,
    WRatio,
]

thresholds = list(range(50, 100, 5))

processors = [
    None,
    hard_normalizer,
    light_normalizer,
    legal_text_normalizer,
    phonetic_normalizer,
]

In [ ]:
top_result = run_evaluation_grid_search(
    scorers=scorers,
    thresholds=thresholds,
    processors=processors,
    document_pairs=document_pairs,
    target_label="PER",
    base_output_path=PRE_CLUSTERIZATION_ROOT / "grid-search-results",
)

In [ ]:
print(
    f"The best hyperparameter combination is '{top_result['name']}' "
    f"with an average score of {top_result['score']:.4f}."
)

In [ ]:
gold_path = PRE_CLUSTERIZATION_ROOT / "gold-jsons"

pred_path = PRE_CLUSTERIZATION_ROOT / "grid-search-results" / top_result["name"]

In [ ]:
result_validation_results, result_validation_avg_score = (
    aymurai_metrics.evaluate_prediction_directories(
        gold_dir=gold_path,
        preds_dir=pred_path,
        target_label="PER",
        pred_json_suffix="-ner-preds-pre-cluster",
        gold_json_suffix="-canonical-entities-gold",
    )
)

In [ ]:
result_validation_avg_score

In [ ]:
result_validation_results

Check one of the pre-clusterization in one of the documents because there is only one cluster

In [ ]:
json_to_check = load_json(json_file_path=document_pairs[1][1])

In [ ]:
pre_cluster_json = cluster_with_cdist(
    items=json_to_check,
    threshold=50,
    scorer=partial_token_set_ratio,
    processor=hard_normalizer,
)

In [ ]:
# Convert to Canonical Entities
canonical_entities = clusters_to_canonical_entities(pre_cluster_json)

canonical_entities = [
    CanonicalEntity.model_validate(entity) for entity in canonical_entities
]

canonical_entities = [
    entity.model_dump() | {"entity_id": entity.entity_id.hex}
    for entity in canonical_entities
]

In [ ]:
canonical_entities

We are now analyzing the similarity matrix to identify items connected to multiple neighbors. By leveraging the Union-Find algorithm, we can determine if these local connections form a chain or tree structure that merges all entities into a single global cluster.

In [ ]:
processor = light_normalizer

item = load_json(document_pairs[1][1])

entities = [item.get("text", "") for item in item]
labels = [item.get("aymurai_label", "UNKNOWN") for item in item]

if processor:
    normed = [processor(e) for e in entities]
else:
    normed = [str(e) for e in entities]

sim = process.cdist(normed, normed, scorer=partial_token_set_ratio, score_cutoff=50)
sim = np.array(sim)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np

# Create a binary version for visualization: 0 if sim <= 0 else 1
viz_matrix = (sim > 0).astype(int)

# Define the colormap: 0 -> white, 1 -> red
my_cmap = ListedColormap(["white", "red"])

mask = np.triu(np.ones_like(sim, dtype=bool))

# Plot the matrix using the binary color logic
plt.figure(figsize=(6, 6))
sns.heatmap(
    viz_matrix,
    cmap=my_cmap,
    mask=mask,
    cbar=False,
    linewidths=0.5,
    linecolor="lightgray",
    square=True,  # Ensures cells are perfectly square
)

plt.title("Connection Visualization (Red if > 0)", fontsize=15)
plt.xlabel("Entity Index")
plt.ylabel("Entity Index")
plt.show()

In [ ]:
item[24]

Based on the similarity matrix, there is a high probability that the algorithm will generate a single "giant cluster" due to a phenomenon known as **chaining**.

Observation of the similarity matrix shows that **item [24]** (identified as an **"L"**) and **item [0]** are almost entirely highlighted in red. This indicates that these entities maintain a similarity score above the threshold with nearly every other entity in the set.
* **Union-Find Logic:** Because the pipeline utilizes a **Union-Find (Disjoint Set Union)** algorithm, a single "bridge" is sufficient to merge two distinct groups. In this case, item [24] acts as a universal bridge.
* **Cluster Collapse:** Since item [24] is connected to almost all other nodes, the algorithm will transitively link every entity into a single tree structure, eventually collapsing the entire dataset into one oversized cluster.

## ***CONCLUSION***: Hyperparameter Optimization Results

After a systematic grid search across various scorers, **`token_set_ratio`** was identified as the top-performing configuration. The following analysis breaks down why this method provided the best balance for our entity disambiguation task:

#### 1. Robustness to Context and Alias Matching (The "Set" Advantage)

Unlike character-level scorers (like `ratio` or `partial_ratio`), `token_set_ratio` treats strings as unordered collections of words (tokens). 

* **Handling Name Aliases:** It is particularly effective at linking different versions of a person's name. For example, it recognizes that **"Juan Pérez"** and **"Juan Alberto Pérez"** refer to the same entity by identifying the shared tokens ("Juan", "Pérez") as a subset, yielding a high similarity score despite the additional middle name.
* **The "Intersection" Logic:** This is crucial for our dataset when a name appears as "Juan Pérez" in one document and "Juan Pérez Asesor/a" in another (where the NER might have accidentally included the role).
* **Subset Resilience:** It yields a score of 100 if one name is a perfect subset of the other, effectively ignoring "noise" words or titles that often appear in raw NER extractions.

#### 2. Superior Performance over `token_sort_ratio`

While `token_sort_ratio` is excellent at handling word reordering (e.g., "Juan Pérez" vs. "Pérez, Juan"), it is highly sensitive to the length of the string. 
* In scenarios involving **aliases with extra names** (e.g., "Juan Pérez" vs. "Juan Alberto Pérez") or **titles** (e.g., "Doctor Juan Pérez"), `token_sort_ratio` would penalize the score significantly because the total word count differs. 
* **`token_set_ratio`** maintains a high similarity score in these cases by prioritizing the common intersection of words over the total string length.

#### 3. Preventing Over-Clustering (Avoidance of "Partial" Aggression)

The grid search revealed that more aggressive scorers, such as `partial_token_set_ratio`, often led to **over-clustering** and data pollution.

* **The Risk of "Partial" Logic:** These scorers are too "patient" with differences. If they find a small, similar fragment within a much longer, unrelated string, they force a high score. 
* **Chaining Effect:** This was the primary cause of the "Chaining Effect" (as seen with **Item [24]**), where a single-letter entity or a common word acts as a universal bridge, transitively merging hundreds of unrelated individuals into a single "giant cluster."
* **The `token_set_ratio` Balance:** It provides the necessary flexibility to catch legitimate aliases without being so aggressive that it collapses the distinct identities of the entire corpus.

### Final Recommendation

The **`token_set_ratio`** achieved the highest **Weighted Disambiguation Score** because it effectively handles variable-length mentions, middle names, and word-order changes while maintaining a strict enough threshold to avoid the transitive merging of unrelated person entities.